Script pour l'extraction de sous images de `Imagenet_full` qui utilise les fichiers de localisation.

https://www.kaggle.com/c/imagenet-object-localization-challenge


In [1]:
from retinotopy import *

HOST='obiwan.local'
Running on metal mps
Welcome on macOS-14.4.1-arm64-arm-64bit
On date 2024-04-25, Running learning on host obiwan.local with device mps


In [2]:
args = Params()
data_set_type = 'full'
args.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing original images
target_data_set_type = 'bbox'
args.target_root = f'{DATAROOT}/Imagenet_{target_data_set_type}' # Directory containing cropped images
os.makedirs(args.target_root, exist_ok=True)
args.folders, args.root, args.target_root

(['val', 'train'],
 '/Volumes/SSD1TO/ImageNet/Imagenet_full',
 '/Volumes/SSD1TO/ImageNet/Imagenet_bbox')

## reading localisation metadata

In [3]:
for folder in args.folders :
    with open(f'{args.root}/LOC_{folder}_solution.csv', 'r') as csv_file:
        df_data = pd.read_csv(csv_file)

In [4]:
df_data

,ImageId,PredictionString
0,n02017213_7894,n02017213 115 49 448 294
1,n02017213_7261,n02017213 91 42 330 432
2,n02017213_5636,n02017213 230 104 414 224
3,n02017213_6132,n02017213 46 82 464 387
4,n02017213_7659,n02017213 103 66 331 335
...,...,...
544541,n02099849_7688,n02099849 0 273 165 498
544542,n02099849_2401,n02099849 123 117 376 374
544543,n02099849_3261,n02099849 0 0 499 332
544544,n02099849_2300,n02099849 151 146 332 333 n02099849 7 232 331 ...


In [5]:
def get_boxes(df, value):
    idx = list(df['ImageId'][df['ImageId'] == value].index)
    bboxes = []
    if idx:
        for i in range(len(df["PredictionString"][idx[0]].split(' '))//5):
            pos =(5*i)
            bboxes.append({'xmin' : int(df["PredictionString"][idx[0]].split(' ')[1 + pos]),
                           'ymin' : int(df["PredictionString"][idx[0]].split(' ')[2 + pos]),
                           'xmax' : int(df["PredictionString"][idx[0]].split(' ')[3 + (5 *i)]),
                           'ymax' : int(df["PredictionString"][idx[0]].split(' ')[4 + (5 *i)])
                                        })
    return bboxes

In [6]:
get_boxes(df_data, 'n02099849_2300')

[{'xmin': 151, 'ymin': 146, 'xmax': 332, 'ymax': 333},
 {'xmin': 7, 'ymin': 232, 'xmax': 331, 'ymax': 467}]

In [7]:

def clean_list(list_dir, patterns=['.DS_Store', '.ipynb_checkpoints']):
    for pattern in patterns:
        if pattern in list_dir: list_dir.remove(pattern)
    return list_dir

## cropping images

In [8]:
# import shutil
from PIL import Image #, ImageDraw, ImageFont # the Pillow module (PIL) is supported by default by TorchVision

verbose = False
verbose = True

def square_box(xmin, ymin, xmax, ymax):
    temp = ((xmax-xmin)-(ymax-ymin))//2 # signed radius
    if temp > 0 :
        ymin -= temp
        ymax += temp
    else:
        xmin += temp
        xmax -= temp
    return xmin, ymin, xmax, ymax


for folder in args.folders :
    # first level
    print(f'\nFolder \"{folder}\"')
    source_folder = os.path.join(args.root, folder)
    boxes_folder = os.path.join(args.target_root, folder)
    os.makedirs(boxes_folder, exist_ok=True)

    # second level
    with open(f'{args.root}/LOC_{folder}_solution.csv', 'r') as csv_file:
        df_data = pd.read_csv(csv_file)
    for i_img, img_id in enumerate(Imagenet_urls_ILSVRC_2016):
        print(f'\nScraping images for id \"{img_id}\" : {labels[i_img]} ')
        target_folder = os.path.join(boxes_folder, img_id) 
        os.makedirs(target_folder, exist_ok=True)

        img_source_folder = os.path.join(source_folder, img_id)
        for imgs in  clean_list(os.listdir(img_source_folder)):
            data_local = os.path.join(img_source_folder, imgs)
            objects = get_boxes(df_data, imgs.split('.')[0])

            original_image = Image.open(data_local, mode='r')
            for i_obj, object in enumerate(objects): #if len(obj)  > 0 :
                xmin = object['xmin']
                ymin = object['ymin']
                xmax = object['xmax']
                ymax = object['ymax']

                crop_image = original_image.crop(square_box(xmin, ymin, xmax, ymax))
                no = '' if i_obj==0 else f'_{i_obj}'
                img_name = imgs.split('.')[0] + no + '.jpg'
                crop_image.save(os.path.join(target_folder, img_name))

            print(f'\r{len(clean_list(os.listdir(target_folder)))} / {len(clean_list(os.listdir(img_source_folder)))}', end="", flush=not verbose)



Folder "val"

Scraping images for id "n01440764" : tench 
52 / 50
Scraping images for id "n01443537" : goldfish 
162 / 50
Scraping images for id "n01484850" : great_white_shark 
56 / 50
Scraping images for id "n01491361" : tiger_shark 
71 / 50
Scraping images for id "n01494475" : hammerhead 
91 / 50
Scraping images for id "n01496331" : electric_ray 
61 / 50
Scraping images for id "n01498041" : stingray 
80 / 50
Scraping images for id "n01514668" : cock 
65 / 50
Scraping images for id "n01514859" : hen 
93 / 50
Scraping images for id "n01518878" : ostrich 
62 / 50
Scraping images for id "n01530575" : brambling 
63 / 50
Scraping images for id "n01531178" : goldfinch 
56 / 50
Scraping images for id "n01532829" : house_finch 
69 / 50
Scraping images for id "n01534433" : junco 
51 / 50
Scraping images for id "n01537544" : indigo_bunting 
55 / 50
Scraping images for id "n01558993" : robin 
60 / 50
Scraping images for id "n01560419" : bulbul 
53 / 50
Scraping images for id "n01580077" : jay 

In [ ]:
target_folder


In [ ]:
%ls {target_folder}
